In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime, UTC
from pyspark.sql.types import *
import uuid

In [0]:
%sql
create schema if not exists e_com_adb.gold_schema;

In [0]:
gold_run_id = str(uuid.uuid4())
run_ts_str = datetime.now(UTC).strftime('%Y-%m-%d %H:%M:%S')
run_date_str = datetime.now(UTC).strftime('%Y-%m-%d')
print(gold_run_id)
print(run_ts_str)
print(run_date_str)

In [0]:
%sql
create table if not exists e_com_adb.gold_schema.processing_control (
    layer string,
    entity_name string,
    last_processed_silver_run_id string,
    last_processed_silver_run_ts timestamp,
    rows_merged bigint,
    run_status string,
    gold_run_id string,
    update_at timestamp
)

In [0]:
def upsert_to_gold(source_df, target_table, join_key):
    if (spark.catalog.tableExists(target_table)):
        dt = DeltaTable.forName(spark, target_table).alias('t')

        (
            dt
            .merge(
                source_df.alias('s'),
                f"s.{join_key} = t.{join_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        (
            source_df
            .write
            .format('delta')
            .saveAsTable(target_table)
        )

In [0]:
def get_last_processed_silver_ts(entity_name):
    ctrl = (
        spark.
        table('e_com_adb.gold_schema.processing_control')
        .where(
            (col('layer') == 'gold') & 
            (col('entity_name') == entity_name) &
            (col('run_status') == 'SUCCESS')
        )
        .orderBy(col('update_at').desc())
        .limit(1)        
    )

    rows = ctrl.collect()
    print(f'get_last_processed_silver_ts: {rows} ({type(rows)})')

    if not rows:
        return None
    else:
        return rows[0]['last_processed_silver_run_ts']



In [0]:
def upsert_gold_control(entity_name, last_processed_silver_run_id, last_processed_silver_run_ts, rows_merged):
    ctrl_df = spark.createDataFrame(
            [(
                'gold',
                entity_name,
                last_processed_silver_run_id,
                last_processed_silver_run_ts,
                rows_merged,
                "SUCCESS",
                gold_run_id,
                datetime.now(UTC)
            )],
            StructType([
                StructField('layer', StringType()),
                StructField('entity_name', StringType()),
                StructField('last_processed_silver_run_id', StringType()),
                StructField('last_processed_silver_run_ts', TimestampType()),
                StructField('rows_merged', IntegerType()),
                StructField('run_status', StringType()),
                StructField('gold_run_id', StringType()),
                StructField('update_at', TimestampType())
            ])
    )

    dt = DeltaTable.forName(spark, 'e_com_adb.gold_schema.processing_control').alias('t')

    (
        dt
        .merge(
            ctrl_df.alias('s'),
            "s.layer = t.layer and s.entity_name = t.entity_name"
        )
        .whenMatchedUpdate(
            set = {
                'last_processed_silver_run_id': 's.last_processed_silver_run_id',
                'last_processed_silver_run_ts': 's.last_processed_silver_run_ts',
                'rows_merged': 's.rows_merged',
                'run_status': 's.run_status',
                'gold_run_id': 's.gold_run_id',
                'update_at': 's.update_at'
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    
    

In [0]:
last_gold_ts = get_last_processed_silver_ts('orders_information')
print(f"Last processed silver timestamp: {last_gold_ts}")

silver_orders_current = spark.read.table("e_com_adb.silver_schema.orders_transformed")
silver_products_current = spark.read.table("e_com_adb.silver_schema.products_transformed")
silver_payments_current = spark.read.table("e_com_adb.silver_schema.payments_transformed")

if last_gold_ts is None:
    changed_orders = silver_orders_current
    changed_products = silver_products_current
    changed_payments = silver_payments_current

else:
    changed_orders = silver_orders_current.filter(col('updated_at') > last_gold_ts)
    changed_products = silver_products_current.filter(col('updated_at') > last_gold_ts)
    changed_payments = silver_payments_current.filter(col('processed_at') > last_gold_ts)

changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print(f"Number of changed orders: {changed_orders_count}")
print(f"Number of changed products: {changed_products_count}")
print(f"Number of changed payments: {changed_payments_count}")



In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()
impacted_from_payments = changed_payments.select("order_id").distinct()
impacted_from_products = (
    changed_products.alias('p')
    .join(silver_orders_current.alias('o'), col('p.product_id') == col('o.product_id'), 'inner')
    .select("o.order_id")
    .distinct()
)

impacted_order_ids = (
    impacted_from_orders
    .union(impacted_from_payments)
    .union(impacted_from_products)
    .distinct()
)

print(f"Impacted Order ids: {impacted_order_ids.count()}")
display(impacted_order_ids.orderBy(col('order_id')))

In [0]:
impacted_order = (
    silver_orders_current.alias('o')
    .join(impacted_order_ids.alias('i'), col('o.order_id') == col('i.order_id'), 'inner')   
    .select('o.*')     
)


gold_delta = (
    impacted_order.alias('o')
    .join(
        silver_products_current.alias('p'),
        col('o.product_id') == col('p.product_id'), 
        'inner'
    )
    .join(
        silver_payments_current.alias('pay'),
        col('pay.order_id') == col('o.order_id'), 
        'inner'
    )
    .select(
        col('o.order_id'),
        col('o.customer_id'),
        col('p.product_id'),
        col('p.product_name'),
        col('p.category'),
        col('p.price').alias('product_price'),
        col('o.order_status'),
        col('o.order_amount'),
        col('pay.payment_id'),
        col('pay.payment_status'),
        col('pay.paid_amount'),
        col('o.order_date'),
        col('o.order_month'),
        col('o.order_year'),
        greatest(
            col('o.updated_at').cast(TimestampType()),
            col('p.updated_at').cast(TimestampType()),
            col('pay.processed_at').cast(TimestampType())
        ).alias('gold_update_ts')
    )
    .dropDuplicates(['order_id'])
    .withColumn(
        "payment_completion_ratio",
        when(col('order_amount') > 0, col('paid_amount')/col('order_amount'))
        .otherwise(lit(0))
    )
    .withColumn(
        "payment_state",
        when(col('order_amount') == 0, lit("Invalid_order_amount"))
        .when(col('payment_completion_ratio') == 0, lit("Unpaid"))
        .when(col('payment_completion_ratio') == 1, lit("Paid"))
        .when(col('payment_completion_ratio') < 1, lit("Partially_paid"))
        .when(col('payment_completion_ratio') > 1, lit("Over_paid"))
    )
    .withColumns({
        'gold_updated_date': to_date(col('gold_update_ts')),
        'gold_run_id': lit(gold_run_id),
    })
)

print('gold_delta_rows: ', gold_delta.count())
display(gold_delta)

In [0]:
if gold_delta.count() > 0:
    upsert_to_gold(gold_delta,'e_com_adb.gold_schema.orders_information', 'order_id')
else:
    print("No new records to insert in gold table")


In [0]:
display(spark.table('e_com_adb.gold_schema.orders_information'))

In [0]:
if not spark.catalog.tableExists('e_com_adb.gold_schema.orders_information_scd2'):    
    scd_df = (
        gold_delta
        .limit(0)
        .withColumns({
            'valid_from_ts': lit(None).cast(TimestampType()),
            'valid_to_ts': lit(None).cast(TimestampType()),
            'is_current': lit(True).cast(BooleanType())
        })
    )

    (scd_df.write.format('delta').saveAsTable('e_com_adb.gold_schema.orders_information_scd2'))
else:
    pass

if (gold_delta.count() > 0):
    delta_scd2 = DeltaTable.forName(spark, 'e_com_adb.gold_schema.orders_information_scd2').alias('t')

    (
        delta_scd2
        .merge(
            gold_delta.alias('s'),
            't.order_id = s.order_id and t.is_current = true'
        )
        .whenMatchedUpdate(
            condition = """
                not(t.order_status <=> s.order_status) or 
                not(t.order_amount <=> s.order_amount) or 
                not(t.paid_amount <=> s.paid_amount) or 
                not(t.payment_id <=> s.payment_id) or 
                not(t.category <=> s.category) or 
                not(t.product_name <=> s.product_name) or
                not(t.product_price <=> s.product_price)
            """,
            set={
                "t.is_current": "false",
                "t.valid_to_ts": "current_timestamp()"
            }
        )
        .whenNotMatchedInsert(  
            values={
                't.order_id': 's.order_id',
                't.customer_id': 's.customer_id',
                't.product_id': 's.product_id',
                't.product_name': 's.product_name',
                't.category': 's.category',
                't.product_price': 's.product_price',
                't.order_status': 's.order_status',
                't.order_amount': 's.order_amount',
                't.payment_id': 's.payment_id',
                't.payment_status': 's.payment_status',
                't.paid_amount': 's.paid_amount',                
                't.order_date': 's.order_date',
                't.order_month': 's.order_month',
                't.order_year': 's.order_year',
                't.gold_update_ts': 's.gold_update_ts',
                't.payment_completion_ratio': 's.payment_completion_ratio',
                't.payment_state': 's.payment_state',
                't.gold_updated_date': 's.gold_updated_date',
                't.gold_run_id': 's.gold_run_id',
                't.is_current': 'true',
                't.valid_from_ts': 'current_timestamp()',
                't.valid_to_ts': 'null'
            }
        )       
        .execute()
    )

In [0]:
display(spark.table('e_com_adb.gold_schema.orders_information_scd2'))

In [0]:
if gold_delta.count() > 0:
    impacted_categories = (
        gold_delta
        .select(col('category'))
        .where(col('category').isNotNull())
        .distinct()
    )

    category_perf_delta = (
        spark.read
        .table('e_com_adb.gold_schema.orders_information')
        .join(impacted_categories, 'category', 'inner')
        .groupBy('category')
        .agg(
            countDistinct('order_id').alias('total_orders'),
            sum(
                when(col('order_amount') > 0, col('order_amount'))
                .otherwise(lit(0.0))
            ).alias('Gross_Merchandise_Value'),
            sum(coalesce(col('paid_amount'), lit(0.0))).alias('Total_Paid_Amount'),
            avg(col('payment_completion_ratio')).alias('Average_Payment_Completion_Ratio'),
            (
                sum(
                    when(col('payment_status') == 'FAILED', lit(1))
                    .otherwise(lit(0))
                ) / count("*")
             ).alias('Payment_Failure_Rate'),
            )
    )

    upsert_to_gold(category_perf_delta, 'e_com_adb.gold_schema.category_performance', 'category')


In [0]:
display(spark.table('e_com_adb.gold_schema.category_performance'))

In [0]:
%sql
create volume if not exists e_com_adb.gold_schema.gold_snapshots_vol;

In [0]:
latest_orders_path = '/Volumes/e_com_adb/gold_schema/gold_snapshots_vol/gold_latest/order_infomation'
latest_category_path = '/Volumes/e_com_adb/gold_schema/gold_snapshots_vol/gold_latest/category_performance'

historical_orders_path = f'/Volumes/e_com_adb/gold_schema/gold_snapshots_vol/gold_snapshots/orders_information/run_date={run_date_str}/run_ts={run_ts_str}'

historical_category_path = f'/Volumes/e_com_adb/gold_schema/gold_snapshots_vol/gold_snapshots/category_performance/run_date={run_date_str}/run_ts={run_ts_str}'

spark.read.table('e_com_adb.gold_schema.orders_information').write.format('delta').mode('overwrite').save(latest_orders_path)
spark.read.table('e_com_adb.gold_schema.category_performance').write.format('delta').mode('overwrite').save(latest_category_path)

spark.read.table('e_com_adb.gold_schema.orders_information').write.format('delta').mode('overwrite').save(historical_orders_path)
spark.read.table('e_com_adb.gold_schema.category_performance').write.format('delta').mode('overwrite').save(historical_category_path)


print(f'Latest Order path : {latest_orders_path}')
print(f'Latest Category path : {latest_category_path}')
print(f'Historical Order path : {historical_orders_path}')
print(f'Historical Category path : {historical_category_path}')

In [0]:
latest_silver_ts = silver_orders_current.agg(max("bronze_ingested_at").alias('mx')).collect()[0]['mx']

latest_silver_run_id = (
    silver_orders_current
    .where(col('bronze_ingested_at') == latest_silver_ts)
    .agg(max(col('silver_run_id')).alias('mx'))    
    .collect()[0]['mx']
) if latest_silver_ts is not None else None

upsert_gold_control(
    'orders_information',
    latest_silver_run_id,
    latest_silver_ts,
    gold_delta.count()
)

display(spark.table('e_com_adb.gold_schema.processing_control'))
